In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import entropy
import shutil


# ============================================================
# PATHS
# ============================================================

file_path = Path(
    r"C:/Users/muskan/OneDrive/Desktop/effisense/data/raw/dataset.csv"
)

output_file = Path(
    r"C:/Users/muskan/OneDrive/Desktop/effisense/data/processed/mouse_features_10k.csv"
)

# Temporary folder containing one file per user
user_folder = file_path.parent / "mouse_users"


# ============================================================
# CONFIGURATION
# ============================================================

WINDOW_SIZE = 50          # 50 mouse events per tuple
STRIDE = 5                # move window by 5 events
SESSION_GAP = 300         # 5 minutes inactivity = new session
IDLE_THRESHOLD = 2.0      # 2 seconds = pause
TARGET_TUPLES = 10_000
CHUNK_SIZE = 100_000


# ============================================================
# CLEAN OLD OUTPUT / TEMPORARY FILES
# Prevents duplicate data if the notebook is rerun
# ============================================================

if output_file.exists():
    output_file.unlink()

if user_folder.exists():
    shutil.rmtree(user_folder)

user_folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# STEP 1: SPLIT LARGE CSV INTO USER-WISE FILES
# We cannot assume the original CSV is sorted by user/time.
# ============================================================

reader = pd.read_csv(
    file_path,
    sep=";",
    chunksize=CHUNK_SIZE,
    low_memory=False
)

for chunk_no, chunk in enumerate(reader):

    print(f"Processing raw chunk {chunk_no + 1}")

    # Create timestamp from date + time
    chunk["timestamp"] = pd.to_datetime(
        chunk["local_date"].astype(str)
        + " "
        + chunk["local_time"].astype(str),
        errors="coerce"
    )

    # Keep only rows having valid user and timestamp
    chunk = chunk.dropna(
        subset=["idman", "timestamp"]
    )

    # Write each user to a separate file
    for user_id, user_df in chunk.groupby("idman"):

        user_file = user_folder / f"user_{user_id}.csv"

        user_df.to_csv(
            user_file,
            mode="a",
            header=not user_file.exists(),
            index=False
        )

print("\nFinished splitting users.")


# ============================================================
# STEP 2: ENTROPY FUNCTION
# ============================================================

def calculate_entropy(series, bins=20):

    series = pd.Series(series).replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    if len(series) < 2:
        return 0.0

    if series.nunique() <= 1:
        return 0.0

    hist, _ = np.histogram(
        series,
        bins=bins
    )

    if hist.sum() == 0:
        return 0.0

    probabilities = hist / hist.sum()

    probabilities = probabilities[
        probabilities > 0
    ]

    return entropy(
        probabilities,
        base=2
    )


# ============================================================
# STEP 3: EXTRACT FEATURES FROM ONE 50-EVENT WINDOW
# ============================================================

def extract_window_features(
    window,
    user_id,
    session_id,
    window_id
):

    # Always work chronologically
    window = (
        window
        .sort_values("timestamp")
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # VALID MOVEMENT DATA
    # --------------------------------------------------------

    movement = window[
        window["distance"].notna()
        &
        window["velocity"].notna()
        &
        window["angle"].notna()
    ].copy()

    if len(movement) < 5:
        return None


    # --------------------------------------------------------
    # VELOCITY
    # --------------------------------------------------------

    velocity = pd.to_numeric(
        movement["velocity"],
        errors="coerce"
    )

    velocity = velocity.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    # Ignore negative velocity values
    velocity = velocity[
        velocity >= 0
    ]

    if len(velocity) < 5:
        return None

    # Remove extreme velocity outliers
    velocity_threshold = velocity.quantile(0.99)

    valid_velocity = velocity[
        velocity <= velocity_threshold
    ]

    if len(valid_velocity) == 0:
        return None


    # --------------------------------------------------------
    # ACCELERATION
    # Calculated using timestamp differences, NOT duration
    # --------------------------------------------------------

    dt = (
        movement["timestamp"]
        .diff()
        .dt.total_seconds()
    )

    # Invalid / zero time differences
    dt = dt.replace(
        [np.inf, -np.inf],
        np.nan
    )

    dt[dt <= 0] = np.nan

    acceleration = (
        movement["velocity"].diff() / dt
    )

    acceleration = acceleration.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    # Remove extreme acceleration outliers
    if len(acceleration) > 2:

        acc_limit = (
            acceleration.abs().quantile(0.99)
        )

        valid_acceleration = acceleration[
            acceleration.abs() <= acc_limit
        ]

    else:

        valid_acceleration = acceleration


    # --------------------------------------------------------
    # DIRECTION CHANGE
    # Angle is treated as degrees
    # --------------------------------------------------------

    angles = pd.to_numeric(
        movement["angle"],
        errors="coerce"
    ).dropna()

    angle_diff = angles.diff().abs()

    # Handle circular angle:
    # e.g. 359° -> 1° should be 2°, not 358°
    angle_diff = np.minimum(
        angle_diff,
        360 - angle_diff
    )

    angle_diff = angle_diff.dropna()


    # --------------------------------------------------------
    # TIME GAPS / PAUSES
    # --------------------------------------------------------

    event_dt = (
        window["timestamp"]
        .diff()
        .dt.total_seconds()
    )

    event_dt = event_dt.replace(
        [np.inf, -np.inf],
        np.nan
    )

    pause_durations = event_dt[
        event_dt > IDLE_THRESHOLD
    ].dropna()


    # --------------------------------------------------------
    # WINDOW DURATION
    # This is window duration, not complete session duration
    # --------------------------------------------------------

    if len(window) > 1:

        window_duration = (
            window["timestamp"].iloc[-1]
            -
            window["timestamp"].iloc[0]
        ).total_seconds()

    else:

        window_duration = 0

    if window_duration < 0:
        window_duration = 0


    # --------------------------------------------------------
    # VELOCITY VARIABILITY
    # --------------------------------------------------------

    mean_velocity = valid_velocity.mean()
    std_velocity = valid_velocity.std()

    if mean_velocity > 0:

        velocity_cv = (
            std_velocity / mean_velocity
        )

        smoothness = (
            1 / (1 + velocity_cv)
        )

    else:

        velocity_cv = 0
        smoothness = 0


    # --------------------------------------------------------
    # FINAL FEATURE TUPLE
    # --------------------------------------------------------

    return {

        "user": user_id,
        "session": session_id,
        "window": window_id,

        # Velocity
        "mean_velocity":
            mean_velocity,

        "std_velocity":
            std_velocity,

        "median_velocity":
            valid_velocity.median(),

        "max_velocity":
            valid_velocity.max(),

        "min_velocity":
            valid_velocity.min(),

        # Acceleration
        "mean_acceleration":
            valid_acceleration.mean()
            if len(valid_acceleration)
            else 0,

        "std_acceleration":
            valid_acceleration.std()
            if len(valid_acceleration)
            else 0,

        "max_acceleration":
            valid_acceleration.abs().max()
            if len(valid_acceleration)
            else 0,

        # Direction
        "mean_direction_change":
            angle_diff.mean()
            if len(angle_diff)
            else 0,

        "median_direction_change":
            angle_diff.median()
            if len(angle_diff)
            else 0,

        # Distance
        "total_distance":
            movement["distance"].sum(),

        "mean_distance":
            movement["distance"].mean(),

        # Time
        "window_duration":
            window_duration,

        # Pauses
        "idle_events":
            len(pause_durations),

        "idle_time":
            pause_durations.sum()
            if len(pause_durations)
            else 0,

        "idle_percentage":
            (
                pause_durations.sum()
                / window_duration
                * 100
            )
            if window_duration > 0
            else 0,

        "pause_count":
            len(pause_durations),

        "mean_pause":
            pause_durations.mean()
            if len(pause_durations)
            else 0,

        "median_pause":
            pause_durations.median()
            if len(pause_durations)
            else 0,

        "std_pause":
            pause_durations.std()
            if len(pause_durations)
            else 0,

        "pause_95":
            pause_durations.quantile(0.95)
            if len(pause_durations)
            else 0,

        # Entropy
        "velocity_entropy":
            calculate_entropy(valid_velocity),

        "direction_entropy":
            calculate_entropy(angle_diff),

        # Behavioral variability
        "velocity_cv":
            velocity_cv,

        "smoothness":
            smoothness,

        # Number of raw events in tuple
        "num_events":
            len(window)
    }


# ============================================================
# STEP 4: CREATE SESSIONS FOR ONE USER
# New session whenever inactivity > 5 minutes
# ============================================================

def create_sessions(df):

    df = (
        df
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    gaps = (
        df["timestamp"]
        .diff()
        .dt.total_seconds()
    )

    df["session_id"] = (
        gaps > SESSION_GAP
    ).cumsum()

    return df


# ============================================================
# STEP 5: PROCESS ONE USER
# User -> sessions -> sliding windows -> feature tuples
# ============================================================

def process_user(
    user_file,
    output_file,
    target_tuples,
    current_count
):

    user_df = pd.read_csv(
        user_file,
        low_memory=False
    )

    # Convert timestamp again after reading user file
    user_df["timestamp"] = pd.to_datetime(
        user_df["timestamp"],
        errors="coerce"
    )

    user_df = user_df.dropna(
        subset=["timestamp"]
    )

    if len(user_df) < WINDOW_SIZE:
        return current_count


    # --------------------------------------------------------
    # SORT THIS USER CHRONOLOGICALLY
    # --------------------------------------------------------

    user_df = (
        user_df
        .sort_values("timestamp")
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # CREATE SESSIONS
    # --------------------------------------------------------

    user_df = create_sessions(
        user_df
    )

    user_id = user_df["idman"].iloc[0]

    results = []


    # --------------------------------------------------------
    # PROCESS SESSION BY SESSION
    # --------------------------------------------------------

    for session_id, session_df in user_df.groupby(
        "session_id"
    ):

        session_df = (
            session_df
            .reset_index(drop=True)
        )

        if len(session_df) < WINDOW_SIZE:
            continue


        window_id = 0


        # ----------------------------------------------------
        # SLIDING WINDOWS
        # 50 events, moving 5 events at a time
        # ----------------------------------------------------

        for start in range(
            0,
            len(session_df) - WINDOW_SIZE + 1,
            STRIDE
        ):

            window = session_df.iloc[
                start:
                start + WINDOW_SIZE
            ]


            features = extract_window_features(
                window,
                user_id,
                session_id,
                window_id
            )


            if features is not None:

                results.append(
                    features
                )

                current_count += 1


            window_id += 1


            # Stop once 10,000 useful tuples are reached
            if current_count >= target_tuples:
                break


        if current_count >= target_tuples:
            break


    # --------------------------------------------------------
    # SAVE RESULTS
    # --------------------------------------------------------

    if results:

        result_df = pd.DataFrame(
            results
        )

        result_df.to_csv(
            output_file,
            mode="a",
            header=not output_file.exists(),
            index=False
        )


    return current_count


# ============================================================
# STEP 6: PROCESS ALL USERS
# Stop after >= 10,000 tuples
# ============================================================

user_files = sorted(
    user_folder.glob("user_*.csv")
)

print(
    f"\nUsers found: {len(user_files)}"
)

current_count = 0


for i, user_file in enumerate(user_files, start=1):

    print(
        f"\nProcessing user {i}/{len(user_files)}: "
        f"{user_file.stem}"
    )

    current_count = process_user(
        user_file,
        output_file,
        TARGET_TUPLES,
        current_count
    )

    print(
        f"Tuples generated: {current_count}"
    )

    if current_count >= TARGET_TUPLES:
        break
# ============================================================
# STEP 7: FINAL CHECK
# ============================================================

if output_file.exists():

    final_df = pd.read_csv(output_file)

    print("\n===================================")
    print("FEATURE EXTRACTION COMPLETE")
    print("===================================")

    print("Final shape:", final_df.shape)
    print("Total tuples:", len(final_df))
    print("Columns:", final_df.columns.tolist())

    # The feature extraction uses "user" as the output column
    if "user" in final_df.columns:
        print("Unique users:", final_df["user"].nunique())

    print("Output:", output_file)

else:

    print("\nNo feature tuples were generated.")




Processing raw chunk 1
Processing raw chunk 2
Processing raw chunk 3
Processing raw chunk 4
Processing raw chunk 5
Processing raw chunk 6
Processing raw chunk 7
Processing raw chunk 8
Processing raw chunk 9
Processing raw chunk 10
Processing raw chunk 11
Processing raw chunk 12
Processing raw chunk 13
Processing raw chunk 14
Processing raw chunk 15
Processing raw chunk 16
Processing raw chunk 17
Processing raw chunk 18
Processing raw chunk 19
Processing raw chunk 20
Processing raw chunk 21
Processing raw chunk 22
Processing raw chunk 23
Processing raw chunk 24
Processing raw chunk 25
Processing raw chunk 26
Processing raw chunk 27
Processing raw chunk 28
Processing raw chunk 29
Processing raw chunk 30
Processing raw chunk 31
Processing raw chunk 32
Processing raw chunk 33
Processing raw chunk 34
Processing raw chunk 35
Processing raw chunk 36
Processing raw chunk 37
Processing raw chunk 38
Processing raw chunk 39
Processing raw chunk 40
Processing raw chunk 41
Processing raw chunk 42
P

KeyError: 'user'

In [23]:
# ============================================================
# STEP 7: FINAL CHECK
# ============================================================

if output_file.exists():

    final_df = pd.read_csv(output_file)

    print("\n===================================")
    print("FEATURE EXTRACTION COMPLETE")
    print("===================================")

    print("Final shape:", final_df.shape)
    print("Total tuples:", len(final_df))
    print("Columns:", final_df.columns.tolist())

    # The feature extraction uses "user" as the output column
    if "user" in final_df.columns:
        print("Unique users:", final_df["user"].nunique())

    print("Output:", output_file)

else:

    print("\nNo feature tuples were generated.")


FEATURE EXTRACTION COMPLETE
Final shape: (9999, 29)
Total tuples: 9999
Columns: ['1', '0', '0.1', '578.5833333333334', '685.5699349550605', '367.0', '3580.0', '101.0', '-597.4420615361843', '2120.227749577184', '10222.32348938496', '42.5', '35.0', '15579.0', '421.05405405405406', '214.425973', '10', '191.21271560000002', '89.17423245177487', '10.1', '19.121271560000004', '17.318468', '21.671362721290876', '52.086937154999944', '2.4734445066183084', '3.7709103354256377', '1.1849113091546486', '0.45768448165839015', '50']
Output: C:\Users\muskan\OneDrive\Desktop\effisense\data\processed\mouse_features_10k.csv
